# NBA Comp Viz — Colab harvest runner
Runs the full harvest chain (build → segment → matchups → align → join → credit)
on a Colab GPU (~4-6× faster than local MPS). The repo is self-contained: code,
model weights, PBP data and the games registry all clone from GitHub.

**Runtime → Change runtime type → GPU (T4)** before running.

Resume-safe: `data/harvest/status.json` is kept on Drive, so a killed session
picks up where it left off — just Run All again.

In [ ]:
#@title 1 · Clone repo + install deps
!git clone --depth 1 https://github.com/lmcnulty7/NBA-Comp-Viz-V3.git repo 2>/dev/null || (cd repo && git pull)
%cd repo
!pip -q install ultralytics easyocr yt-dlp 2>&1 | tail -1
import torch; print("device:", "cuda" if torch.cuda.is_available() else "CPU ONLY — enable GPU runtime!")

In [ ]:
#@title 2 · Mount Drive (persistence for videos, status, results)
from google.colab import drive
drive.mount('/content/drive')
import os
PERSIST = "/content/drive/MyDrive/nba_harvest"
os.makedirs(f"{PERSIST}/video", exist_ok=True)
os.makedirs("data/harvest", exist_ok=True)
# link the video store + carry over any previous session's status (resume)
!ln -sfn {PERSIST}/video data/harvest/video
!test -f {PERSIST}/status.json && cp {PERSIST}/status.json data/harvest/status.json || true
print("persist dir ready")

In [ ]:
#@title 3 · Download games (skips ones already on Drive)
GAMES = ["gsw_cle_2017f_g5", "gsw_sac_klay37", "gsw_nyk_curry54",
         "gsw_mia_2017", "gsw_splash62", "gsw_hou_duel"]
import json, subprocess, os
reg = json.load(open("data/harvest/games.json"))
for t in GAMES:
    dst = f"data/harvest/video/{t}.mp4"
    if os.path.exists(dst) and os.path.getsize(dst) > 1e8:
        print(t, "already on Drive"); continue
    vid = reg[t]["video_id"]
    r = subprocess.run(["yt-dlp", "-f", "bv*[height<=720][height>=480]+ba",
                        "--merge-output-format", "mp4", "-o", dst, "-q", "--no-warnings",
                        f"https://www.youtube.com/watch?v={vid}"])
    print(t, "OK" if r.returncode == 0 else "FAILED — if YouTube blocks Colab IPs, "
          "upload the mp4s from your laptop to Drive:nba_harvest/video/ instead")

In [ ]:
#@title 4 · Run the harvest queue (resume-safe; ~30-45 min/game on T4)
import subprocess, shutil
r = subprocess.run(["python", "harvest_driver.py", "--games"] + GAMES + ["--no-align"])
shutil.copy("data/harvest/status.json", f"{PERSIST}/status.json")   # persist resume state
print("driver exit", r.returncode)

In [ ]:
#@title 5 · Align + join + credit (CPU-light, GPU OCR)
import json, subprocess, shutil
st = json.load(open("data/harvest/status.json"))
secs = sorted(c for c, u in st.items() if "_s" in c and u.get("segment", {}).get("state") == "ok"
              and any(c.startswith(t) for t in GAMES))
print(len(secs), "sections to align")
subprocess.run(["python", "align_outcomes.py", "--clips"] + secs)
for c in secs:
    subprocess.run(["python", "matchup_metrics.py", "--clip", c, "--no-video"],
                   capture_output=True)
subprocess.run(["python", "tier2_join.py"])
subprocess.run(["python", "tier2_credit.py"])
shutil.copy("data/harvest/status.json", f"{PERSIST}/status.json")

In [ ]:
#@title 6 · Package results → Drive (JSONs only, a few hundred MB max)
!mkdir -p {PERSIST}/results
!zip -qr {PERSIST}/results/night3_tracking.zip data/tracking -i "data/tracking/gsw_*"
!zip -qr {PERSIST}/results/night3_pbp.zip data/pbp
!cp data/harvest/status.json {PERSIST}/results/
print("results on Drive: nba_harvest/results/ — download night3_tracking.zip + night3_pbp.zip,")
print("unzip into the local repo, then run tier2_join.py + tier2_credit.py locally (or read the zipped ones).")